# 01. Vectorless RAG with BM25

> *Module 01 was vector RAG: find documents whose embeddings are closest to your query. Module 02 was graph RAG: find documents you can reach by following relationships. This module is the older, simpler, often faster baseline that most people skip past on the way to vectors: BM25.*

Three notebooks. **This is the crawl.** A hand-written three-doc corpus inline, the BM25 scoring intuition in one diagram, the `bm25s` API in five lines, then a real query against the seed corpus where we can hand-verify the ranking. No LLM, no embedding model, no neural network of any kind. Just word counting with statistics.

## The scoring intuition

BM25 ranks documents by a score that's the product of three pieces:

```
              ┌────────────┐   ┌──────────────────┐   ┌─────────────────────┐
score(doc) =  │    IDF     │ × │ TF saturation    │ × │ length normalization │
              │ rare words │   │ more matches     │   │ shorter docs win    │
              │ count more │   │ count more, but  │   │ ties (length-       │
              │            │   │ with diminishing │   │ adjusted)           │
              │            │   │ returns          │   │                     │
              └────────────┘   └──────────────────┘   └─────────────────────┘
```

That's the whole algorithm. Everything fancier is plumbing.

- **IDF (inverse document frequency)** rewards rare words. If a query token appears in 1 of 10,000 docs, finding it is a strong signal; if it appears in 9,000 of 10,000 docs, it's almost noise.
- **TF saturation** rewards repeated matches, but with diminishing returns. A doc that has the query word twice is much better than one that has it zero times, but only slightly better than one that has it ten times.
- **Length normalization** gives shorter docs a small edge over longer ones at the same raw TF. Otherwise a long document would always win on raw count just by mentioning everything once.

## First contact: three documents

The smallest possible BM25 demo. Three documents, hand-written so you can see the algorithm work without any other noise.

In [1]:
import bm25s

docs = [
    "cats are small furry animals that like fish",
    "dogs are loyal furry animals that come when called",
    "pythons are reptiles, not the programming language",
]

tokens = bm25s.tokenize(docs, show_progress=False)
print(tokens)

Tokenized(
  "ids": [
    0: [0, 1, 2, 3, 4, 5]
    1: [6, 7, 2, 3, 8, 9, 10]
    2: [11, 12, 13, 14]
  ],
  "vocab": [
    'animals': 3
    'called': 10
    'cats': 0
    'come': 8
    'dogs': 6
    'fish': 5
    'furry': 2
    'language': 14
    'like': 4
    'loyal': 7
    ... (total 15 tokens)
  ],
)


`tokenize` returns a `Tokenized` object holding the corpus tokens. By default it lowercases, strips punctuation, and removes common English stopwords ("are", "the", "that"). Notice in the printed output that "are", "the", and "that" are gone, and capitalization has been folded. We'll dig into tokenization in NB2.

In [2]:
retriever = bm25s.BM25()
retriever.index(tokens, show_progress=False)
print(f"Indexed {len(docs)} documents")

Indexed 3 documents


## Your first query

In [3]:
query = bm25s.tokenize("furry animals", show_progress=False)
results, scores = retriever.retrieve(query, k=3, show_progress=False)

for rank, (idx, score) in enumerate(zip(results[0], scores[0]), start=1):
    print(f"  #{rank}  score={float(score):.3f}  {docs[idx]}")

  #1  score=0.366  cats are small furry animals that like fish
  #2  score=0.340  dogs are loyal furry animals that come when called
  #3  score=0.000  pythons are reptiles, not the programming language


The two animal documents win, the python document loses. The animals docs both match "furry" *and* "animals", the python doc matches neither. The scores are positive but small because both query terms are common across the matching documents (low IDF). If the query were `"reptiles"` instead, the python doc would spike to the top because "reptiles" appears in exactly one of the three (high IDF). Hand-verify that:

In [4]:
query = bm25s.tokenize("reptiles", show_progress=False)
results, scores = retriever.retrieve(query, k=3, show_progress=False)
for rank, (idx, score) in enumerate(zip(results[0], scores[0]), start=1):
    print(f"  #{rank}  score={float(score):.3f}  {docs[idx]}")

  #1  score=0.452  pythons are reptiles, not the programming language
  #2  score=0.000  dogs are loyal furry animals that come when called
  #3  score=0.000  cats are small furry animals that like fish


The first result's score is much higher than anything in the previous query, even though both queries had one token. That's IDF in action: "reptiles" appears in 1 of 3 docs, "furry" appears in 2 of 3. The rarer token rewards its match more.

## A real corpus

Now we load the 12-tip Python standard library corpus from `scripts/build_corpus.py`. Twelve short documents, each on one topic. Small enough to print and read through in a minute, big enough that IDF and TF saturation behave like they do on real corpora.

In [5]:
import pandas as pd
from scripts.build_corpus import build_if_missing

build_if_missing()
df = pd.read_parquet("data/corpus.parquet")
df[["id", "title"]]

,id,title
0,tip_01,Sort a list of dicts by a key with operator.it...
1,tip_02,Sort with a custom key function
2,tip_03,bisect keeps a sorted list sorted on insert
3,tip_04,collections.OrderedDict preserves insertion order
4,tip_05,dict itself preserves insertion order since Py...
5,tip_06,Use __slots__ to shrink instance memory
6,tip_07,Walrus operator := for expression assignment
7,tip_08,dataclasses.dataclass removes __init__ boilerp...
8,tip_09,functools.lru_cache memoizes pure functions
9,tip_10,pathlib.Path replaces os.path for new code


In [6]:
tokens = bm25s.tokenize(df["text"].tolist(), show_progress=False)
retriever = bm25s.BM25()
retriever.index(tokens, show_progress=False)
print(f"Indexed {len(df)} documents")

Indexed 12 documents


## Query 1: a literal vocabulary hit

`"how do I sort a list of dicts by name"` should land squarely on tip_01. The doc uses every key word.

In [7]:
def show_top(retriever, df, query: str, k: int = 3):
    q_tokens = bm25s.tokenize(query, show_progress=False)
    results, scores = retriever.retrieve(q_tokens, k=k, show_progress=False)
    for rank, (idx, score) in enumerate(zip(results[0], scores[0]), start=1):
        row = df.iloc[idx]
        print(f"  #{rank}  {row['id']}  score={float(score):.3f}  {row['title']}")

show_top(retriever, df, "how do I sort a list of dicts by name")

  #1  tip_01  score=2.973  Sort a list of dicts by a key with operator.itemgetter
  #2  tip_03  score=1.523  bisect keeps a sorted list sorted on insert
  #3  tip_02  score=1.308  Sort with a custom key function


tip_01 wins with a clear margin. The doc has "sort", "list", "dicts", and discusses sorting by a field with `operator.itemgetter`. tip_03 ranks second because it contains "sorted list" and "sort" and "sorting"; tip_02 ranks third because of the shared "sort" + "key" vocabulary. The math matches the intuition.

## Query 2: the rare-keyword spike

`"binary search insert"` doesn't share many tokens with most of the corpus, but it shares "insert" and the concept of "search" with tip_03 (about `bisect`). The IDF reward on those rare tokens spikes the score.

In [8]:
show_top(retriever, df, "binary search insert")

  #1  tip_03  score=0.899  bisect keeps a sorted list sorted on insert
  #2  tip_12  score=0.000  Speed up loops by avoiding global name lookups
  #3  tip_10  score=0.000  pathlib.Path replaces os.path for new code


tip_03 (`bisect`) wins. The second and third results both score zero, meaning they share no non-stopword tokens with the query at all. **This kind of rare-keyword query is exactly where BM25 beats vector retrieval.** Vector embeddings tend to smear "binary search insert" together with "search" and "insert" generally, ranking too many distantly related docs. BM25 just nails the one with the right rare words.

Hold onto that observation. NB3's head-to-head against module 01's Pinecone index is built around it.

## A taste of where this breaks

Try the same query with a one-word swap: "dictionaries" instead of "dicts".

In [9]:
show_top(retriever, df, "how do I sort a list of dictionaries")

  #1  tip_03  score=1.523  bisect keeps a sorted list sorted on insert
  #2  tip_01  score=1.364  Sort a list of dicts by a key with operator.itemgetter
  #3  tip_02  score=1.308  Sort with a custom key function


tip_01 doesn't win anymore. The doc has "dicts", not "dictionaries", and BM25 doesn't know they mean the same thing. This is the **vocabulary-mismatch problem**: a perfectly semantic query that fails because of a one-token spelling difference.

NB2 looks at the tools BM25 has for this (stemming and tokenization options). NB3 looks at the tools it doesn't (vector embeddings, hybrid retrieval). For now, just notice it.

## Recap

- BM25 ranks documents by IDF × TF saturation × length normalization. Three statistics multiplied together.
- The `bm25s` API has five calls: `tokenize`, `BM25()`, `index`, `tokenize` again for the query, `retrieve`.
- Rare query tokens reward big. Common ones reward small. Repeated matches help, but with diminishing returns.
- The biggest failure mode is vocabulary mismatch: synonyms, paraphrases, words your document doesn't literally contain.

**Next:** [NB2 - 02. BM25 Mechanics](./02_bm25_mechanics.ipynb) opens the knobs. Tokenization options, the `k1` and `b` parameters, where BM25 wins and where it loses, and how to persist an index.